# 02 · `[컴바인 단독]` 전처리 · 이상치 정제 · 결측(포기 선수) 구제 · 피처 추출

이 노트북은 **컴바인 데이터(`combine_results.csv`, `combine_tracking.csv`, `players.csv` 사전 정보)**만 다루며, **1셀 = 1작업 단계** 원칙으로 아래 흐름을 따라간다:
1. 데이터 로딩 → 2. 데이터 정제(`dis` 오염 폐기) → 3. 이상치 탐지·처리 → 4. 결측치 탐지 & 포기 선수 실태 → 5. 피처 추가(3축 운동학·40야드 F-V·스킬 드릴 구제) → 6. 피처 추가(체격·물리·나이·서브포지션) → 7. 검증 및 저장

### [0] 실행 환경 부트스트랩 (로컬 / Colab 겸용)
- **데이터 범위**: `공통 환경`
- **작업 단계**: `환경 설정`
- **목적 및 원리**: `StopIteration` 없이 프로젝트 루트를 찾고 `src/config.py`를 연결한다.
- **확인할 내용**: `9/9 CSVs ready`.

In [ ]:
# =====================================================================
# [데이터: 공통] [단계: 환경 설정] 로컬/VS Code Colab 커널 겸용 루트 탐색
# =====================================================================
import importlib.util
import os
import sys
from pathlib import Path

# Google Drive 내 다른 경로에 저장소/데이터가 있다면 아래 주석을 풀고 경로를 지정하세요.
# os.environ["NFL_PROJECT_ROOT"] = "/content/drive/MyDrive/nfl-bdb2027"
# os.environ["NFL_DATA_DIR"] = "/content/drive/MyDrive/nfl-bdb2027-data"

def _is_colab_kernel() -> bool:
    if (
        "google.colab" in sys.modules
        or "COLAB_RELEASE_TAG" in os.environ
        or "COLAB_GPU" in os.environ
    ):
        return True
    try:
        return importlib.util.find_spec("google.colab") is not None
    except (ModuleNotFoundError, ValueError):
        return False

def find_project_root() -> Path:
    candidates: list[Path] = []
    if env_root := os.environ.get("NFL_PROJECT_ROOT"):
        candidates.append(Path(env_root).expanduser())

    cwd = Path.cwd()
    candidates.extend([cwd, *cwd.parents, cwd / "nfl-bdb2027"])

    in_colab = _is_colab_kernel()
    drive_root = Path("/content/drive")
    mydrive = drive_root / "MyDrive"

    if in_colab:
        if not mydrive.exists():
            try:
                from google.colab import drive
                drive.mount(str(drive_root), force_remount=False)
            except Exception as exc:
                print(f"[warn] Google Drive 자동 마운트 실패: {exc}")

        candidates.extend([
            Path("/content/nfl-bdb2027"),
            mydrive / "nfl-bdb2027",
            mydrive / "Colab Notebooks" / "nfl-bdb2027",
        ])

    seen: set[Path] = set()
    for p in candidates:
        if p in seen:
            continue
        seen.add(p)
        if (p / "src" / "config.py").exists():
            return p.resolve()

    if in_colab:
        skip = {"drive", "sample_data", ".config", ".git", ".venv", "__pycache__"}
        depth1_dirs: list[Path] = []
        for base in (Path("/content"), mydrive, drive_root / "Shareddrives"):
            if not base.exists():
                continue
            try:
                children = [
                    d for d in base.iterdir()
                    if d.is_dir() and d.name not in skip and not d.name.startswith(".")
                ]
            except OSError:
                continue
            depth1_dirs.extend(children)
            for d in children:
                for cand in (d, d / "nfl-bdb2027"):
                    if (cand / "src" / "config.py").exists():
                        return cand.resolve()

        for d in depth1_dirs:
            try:
                for sub in d.iterdir():
                    if not sub.is_dir() or sub.name.startswith("."):
                        continue
                    low = sub.name.lower()
                    if "nfl" in low or "bdb" in low:
                        if (sub / "src" / "config.py").exists():
                            return sub.resolve()
                    elif low in {"ml", "work", "projects", "repos", "github", "colab", "colab notebooks", "study"}:
                        cand = sub / "nfl-bdb2027"
                        if (cand / "src" / "config.py").exists():
                            return cand.resolve()
            except OSError:
                continue

    diag = [f"src/config.py를 찾지 못했습니다 (현재 커널 cwd={cwd}, colab={in_colab})."]
    if in_colab:
        for base in (Path("/content"), mydrive):
            if base.exists():
                try:
                    entries = sorted(p.name for p in base.iterdir() if not p.name.startswith("."))[:20]
                except OSError:
                    entries = ["<접근 불가>"]
                diag.append(f"  - {base} 폴더 목록: {entries}")
        diag.extend([
            "",
            "[원인] VS Code에서 Colab 커널(ipykernel)을 선택하면 코드는 로컬 PC가 아니라",
            "       원격 Google Colab VM(/content)에서 실행되므로 로컬 폴더를 직접 읽을 수 없습니다.",
            "[해결 방법]",
            "  1) 로컬 PC 파일로 실행: VS Code 우측 상단 커널을 Colab 대신 로컬 '.venv'로 변경",
            "  2) Colab 커널로 실행: Google Drive(/content/drive/MyDrive/nfl-bdb2027)에 저장소를 두거나,",
            '     다른 Drive 경로에 있다면 이 셀 상단에 os.environ["NFL_PROJECT_ROOT"] = "/content/drive/MyDrive/..." 지정',
        ])
    else:
        diag.append("환경변수 NFL_PROJECT_ROOT를 지정하거나 프로젝트 폴더 안에서 실행하세요.")
    raise FileNotFoundError("\n".join(diag))

ROOT = find_project_root()
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src import config
from src.features import combine_fe
config.setup()

### [1] 컴바인 원본 데이터 로딩
- **데이터 범위**: `컴바인 단독`
- **작업 단계**: `데이터 로딩`
- **목적 및 원리**: 드래프트 **전** 예측 원칙을 지키기 위해 `players.csv`에서 `draft_round` 등 드래프트 결과 컬럼은 아예 읽지 않는다.
- **확인할 내용**: 선수 510명, 컴바인 트래킹 463,189프레임.

In [ ]:
# =====================================================================
# [데이터: 컴바인 단독] [단계: 데이터 로딩] 드래프트 결과 컬럼 제외 로딩
# =====================================================================
import numpy as np
import pandas as pd

cr, ct, pl = combine_fe.load_combine_raw()
assert not set(config.DRAFT_FORBIDDEN_FEATURE_COLS).intersection(pl.columns)
display(cr.head(3))

### [2] 트래킹 정제 · `dis` 오염 폐기 · 글리치 탐지 · 3축 가속 분해
- **데이터 범위**: `컴바인 단독`
- **작업 단계**: `데이터 정제` + `이상치 탐지·처리`
- **목적 및 원리**:
  1. `combine_tracking.csv`의 `dis`는 실제 `(x, y)` 거리의 ~53%만 기록되는 오염이 있어 폐기하고 `hypot(dx, dy)`로 재계산한다.
  2. `a`는 부호 없는 총 가속도 크기이므로 추진($a_\parallel^+$), 제동($a_\parallel^-$), 구심/횡가속($a_\perp = s|\omega|$) 3축으로 분해한다.
- **확인할 내용**: `raw dis / 실제 거리` 중앙비(~0.53) 및 글리치 프레임 수(0건 예상).

In [ ]:
# =====================================================================
# [데이터: 컴바인 단독] [단계: 데이터 정제 & 이상치 탐지] 3축 운동학 계산
# =====================================================================
ply, clean_stats = combine_fe.clean_and_compute_frame_kinematics(ct)
display(ply[["nfl_id", "drill_type", "s", "a", "step_dist", "a_tan_pos", "a_tan_neg", "a_lat", "jerk"]].dropna().head(5))

### [3] 트라이얼 단위 생체역학 집계 & 40야드 Force-Velocity 곡선 피팅
- **데이터 범위**: `컴바인 단독`
- **작업 단계**: `피처 추가` + `이상치 탐지(트라이얼 잘림 플래그)`
- **목적 및 원리**:
  - 각 트라이얼(`TRIAL_KEY`)의 최고속도·추진·제동·구심 민첩성·경로효율(`<= 1.0`)을 계산하고, 중앙값 프레임의 50% 미만인 잘린 트라이얼에 `flag_truncated`를 붙인다.
  - 40야드 달리기에 Samozino-Morin 지수 곡선 $v(t)=v_{\max}(1-e^{-t/\tau})$를 피팅해 이론 최고속도($v_{\max}$), 초기 폭발력($a_0=v_{\max}/\tau$), 최대 파워($P_{\max}$)를 추출한다.
- **확인할 내용**: 총 6,310개 트라이얼, `path_efficiency <= 1.0`, 40야드 F-V 피팅 $R^2$ 중앙값.

In [ ]:
# =====================================================================
# [데이터: 컴바인 단독] [단계: 피처 추가] 트라이얼 운동학 + 40야드 F-V
# =====================================================================
trials, flags, curves = combine_fe.extract_trial_table(ply)
display(trials.groupby("drill_family")[["s_max", "a_tan_pos_max", "a_tan_neg_max", "a_lat_max", "path_efficiency"]].median().round(3))

### [4] 선수 단위 피처 통합 · 포기(결측) 선수 구제 검증 · Parquet 저장
- **데이터 범위**: `컴바인 단독`
- **작업 단계**: `결측치 탐지` + `피처 추가(도메인 물리 + 스킬 드릴 구제)` + `검증`
- **목적 및 원리**:
  - 체격 비율(`bmi`, `wing_to_height`), 물리 지수(`speed_score`, `burst_score`, `sayers_power_w`), 인적 메타(`draft_age`, `sub_position`, `is_power5`)를 생성한다.
  - 전통 드릴을 포기(결측)한 선수라도 100% 보유한 **포지션 스킬 드릴 트래킹 운동학 + 결측 지시자**를 결합해 실력 있는 포기 선수를 구제할 기반을 만든다.
- **확인할 내용**: `(x, y)` 재계산 `t_10yd` vs 공식 `ten_yd_split` 상관계수, 7개 드릴 전부 포기 선수의 트래킹 구제율(100%).

In [ ]:
# =====================================================================
# [데이터: 컴바인 단독] [단계: 피처 추가 & 검증] T2 생성 및 저장
# =====================================================================
player_fe = combine_fe.build_combine_player_features(cr, pl, trials)

trials.to_parquet(config.PROCESSED_FILES["combine_trials"], index=False)
flags.to_parquet(config.PROCESSED_FILES["combine_trial_flags"], index=False)
curves.to_parquet(config.PROCESSED_FILES["combine_forty_curves"], index=False)
player_fe.to_parquet(config.PROCESSED_FILES["combine_player_features"], index=False)

print("[확인된 결과] 7개 드릴 전부 포기 선수 중 스킬/컴바인 트래킹 구제 예시:")
display(player_fe[player_fe["n_missing_drills"] == 7][[
    "display_name", "combine_position", "sub_position", "draft_age",
    "skill_smax_max", "skill_a_pos_max", "skill_decel_max", "skill_a_lat_max"
]].head(8))